# Chapter 7: TensorFlow Companion

This notebook is the TensorFlow companion for Chapter 7.
It mirrors the TensorFlow section in the book while keeping the code fully
runnable from top to bottom.

The main Chapter 7 walkthrough remains in
`notebooks/chapter-7/chapter-7-deep-learning-for-soccer-analytics.ipynb`.


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {bool(tf.config.list_physical_devices('GPU'))}")


## Generate Simulated Soccer Data

This notebook uses the same simulated-match setup as the rest of the
Chapter 7 companion material.


In [ ]:
n_matches = 1000
rng = np.random.default_rng(42)

possession_home = np.clip(rng.normal(50, 12, n_matches), 30, 70)

match_df = pd.DataFrame(
    {
        "shots_home": rng.poisson(12, n_matches),
        "possession_home": possession_home,
        "xg_home": rng.gamma(2.0, 0.6, n_matches),
        "corners_home": rng.poisson(5, n_matches),
        "fouls_home": rng.poisson(10, n_matches),
        "shots_away": rng.poisson(10, n_matches),
        "possession_away": 100 - possession_home,
        "xg_away": rng.gamma(1.5, 0.6, n_matches),
        "corners_away": rng.poisson(4, n_matches),
        "fouls_away": rng.poisson(11, n_matches),
    }
)

score_signal = (
    0.3 * (match_df["xg_home"] - match_df["xg_away"])
    + 0.02 * (match_df["shots_home"] - match_df["shots_away"])
    + 0.01 * (match_df["possession_home"] - 50)
)
win_probability = 1 / (1 + np.exp(-score_signal))
match_df["home_win"] = (rng.random(n_matches) < win_probability).astype(int)

result_signal = score_signal + rng.normal(0, 0.5, n_matches)
match_df["FTR"] = np.select(
    [result_signal > 0.5, result_signal < -0.5],
    ["H", "A"],
    default="D",
)
match_df["FTHG"] = rng.poisson(np.clip(match_df["xg_home"].to_numpy(), 0.1, None))
match_df["FTAG"] = rng.poisson(np.clip(match_df["xg_away"].to_numpy(), 0.1, None))

feature_cols = [
    "shots_home",
    "possession_home",
    "xg_home",
    "corners_home",
    "fouls_home",
    "shots_away",
    "possession_away",
    "xg_away",
    "corners_away",
    "fouls_away",
]

scaler = StandardScaler()
X = scaler.fit_transform(match_df[feature_cols]).astype("float32")

print(match_df.head())
print(f"\nDataset shape: {match_df.shape}")


## Binary Outcomes: Will the Home Team Win?

### Data Preparation


In [ ]:
y_binary = match_df["home_win"].astype("float32").to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42
)

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(1024, seed=42).batch(32)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(32)

print(f"Training set: {len(X_train)} matches")
print(f"Test set: {len(X_test)} matches")


### Model Architecture


In [ ]:
binary_model = keras.Sequential(
    [
        layers.Input(shape=(X_train.shape[1],)),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ]
)

binary_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=["accuracy", keras.metrics.AUC(name="auc")],
)

binary_model.summary()


### Training and Evaluation


In [ ]:
early_binary = keras.callbacks.EarlyStopping(
    monitor="val_auc", mode="max", patience=10, restore_best_weights=True
)

history_binary = binary_model.fit(
    train_ds,
    epochs=100,
    validation_data=test_ds,
    callbacks=[early_binary],
    verbose=0,
)

loss, acc, auc = binary_model.evaluate(test_ds, verbose=0)
print(f"Test loss: {loss:.4f}")
print(f"Test accuracy: {acc:.3f}")
print(f"Test AUC: {auc:.3f}")


In [ ]:
binary_probs = binary_model.predict(X_test, verbose=0).squeeze()
print(np.round(binary_probs[:10], 3))


## Match Results: Win, Draw, or Loss

### Data Preparation


In [ ]:
label_encoder = LabelEncoder()
y_multiclass = label_encoder.fit_transform(match_df["FTR"])

X_train_mc, X_test_mc, y_train_mc, y_test_mc = train_test_split(
    X, y_multiclass, test_size=0.2, random_state=42
)

train_ds_mc = tf.data.Dataset.from_tensor_slices(
    (X_train_mc, y_train_mc.astype("int32"))
)
train_ds_mc = train_ds_mc.shuffle(1024, seed=42).batch(32)
test_ds_mc = tf.data.Dataset.from_tensor_slices(
    (X_test_mc, y_test_mc.astype("int32"))
).batch(32)

print(label_encoder.classes_)


### Model Architecture


In [ ]:
multiclass_model = keras.Sequential(
    [
        layers.Input(shape=(X_train_mc.shape[1],)),
        layers.Dense(16, activation="relu"),
        layers.Dense(3, activation="softmax"),
    ]
)

multiclass_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

multiclass_model.summary()


### Training and Evaluation


In [ ]:
history_multiclass = multiclass_model.fit(
    train_ds_mc,
    epochs=100,
    validation_data=test_ds_mc,
    verbose=0,
)

loss_mc, acc_mc = multiclass_model.evaluate(test_ds_mc, verbose=0)
print(f"Test loss: {loss_mc:.4f}")
print(f"Test accuracy: {acc_mc:.3f}")


In [ ]:
multiclass_probs = multiclass_model.predict(X_test_mc, verbose=0)
predicted_classes = multiclass_probs.argmax(axis=1)
predicted_labels = label_encoder.inverse_transform(predicted_classes)

print(predicted_labels[:10])


## Goal Prediction: Modeling Scoring Output

### Data Preparation


In [ ]:
y_home_goals = match_df["FTHG"].astype("float32").to_numpy()

X_train_goals, X_test_goals, y_train_goals, y_test_goals = train_test_split(
    X, y_home_goals, test_size=0.2, random_state=42
)

train_ds_goals = tf.data.Dataset.from_tensor_slices((X_train_goals, y_train_goals))
train_ds_goals = train_ds_goals.shuffle(1024, seed=42).batch(32)
test_ds_goals = tf.data.Dataset.from_tensor_slices((X_test_goals, y_test_goals)).batch(32)

print(f"Average home goals: {y_train_goals.mean():.2f}")


### Model Architecture


In [ ]:
goals_model = keras.Sequential(
    [
        layers.Input(shape=(X_train_goals.shape[1],)),
        layers.Dense(32, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="softplus"),
    ]
)

goals_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.Poisson(),
    metrics=["mae"],
)

goals_model.summary()


### Training and Evaluation


In [ ]:
early_goals = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=20, restore_best_weights=True
)

history_goals = goals_model.fit(
    train_ds_goals,
    epochs=200,
    validation_data=test_ds_goals,
    callbacks=[early_goals],
    verbose=0,
)

loss_goals, mae_goals = goals_model.evaluate(test_ds_goals, verbose=0)
print(f"Test loss: {loss_goals:.4f}")
print(f"Test MAE: {mae_goals:.3f}")


In [ ]:
goal_predictions = goals_model.predict(X_test_goals, verbose=0).squeeze()
print(np.round(goal_predictions[:10], 2))


## Full Match Simulation: From Goals to Outcomes

### Data Preparation


In [ ]:
y_away_goals = match_df["FTAG"].astype("float32").to_numpy()

(
    X_train_mtl,
    X_test_mtl,
    y_home_train,
    y_home_test,
    y_away_train,
    y_away_test,
) = train_test_split(X, y_home_goals, y_away_goals, test_size=0.2, random_state=42)

train_ds_mtl = tf.data.Dataset.from_tensor_slices(
    (
        X_train_mtl,
        {"home_goals": y_home_train, "away_goals": y_away_train},
    )
)
train_ds_mtl = train_ds_mtl.shuffle(1024, seed=42).batch(32)
test_ds_mtl = tf.data.Dataset.from_tensor_slices(
    (
        X_test_mtl,
        {"home_goals": y_home_test, "away_goals": y_away_test},
    )
).batch(32)


### Model Architecture


In [ ]:
class MultiTaskMLP(keras.Model):
    def __init__(self, input_size, hidden_size, shared_size):
        super().__init__()
        self.shared_layers = keras.Sequential(
            [
                layers.Dense(hidden_size, activation="relu"),
                layers.Dense(shared_size, activation="relu"),
            ]
        )
        self.home_head = keras.Sequential(
            [
                layers.Dense(32, activation="relu"),
                layers.Dense(1, activation="softplus"),
            ]
        )
        self.away_head = keras.Sequential(
            [
                layers.Dense(32, activation="relu"),
                layers.Dense(1, activation="softplus"),
            ]
        )

    def call(self, x):
        features = self.shared_layers(x)
        home_pred = self.home_head(features)
        away_pred = self.away_head(features)
        return home_pred, away_pred


def build_multitask_model(input_size, hidden_size=64, shared_size=32):
    inputs = keras.Input(shape=(input_size,))
    base_model = MultiTaskMLP(input_size, hidden_size, shared_size)
    home_pred, away_pred = base_model(inputs)
    home_pred = layers.Activation("linear", name="home_goals")(home_pred)
    away_pred = layers.Activation("linear", name="away_goals")(away_pred)
    return keras.Model(
        inputs=inputs,
        outputs={"home_goals": home_pred, "away_goals": away_pred},
    )

multitask_model = build_multitask_model(X_train_mtl.shape[1])
multitask_model.summary()


### Training and Evaluation


In [ ]:
multitask_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss={
        "home_goals": keras.losses.Poisson(),
        "away_goals": keras.losses.Poisson(),
    },
    metrics={
        "home_goals": [keras.metrics.MeanAbsoluteError(name="mae")],
        "away_goals": [keras.metrics.MeanAbsoluteError(name="mae")],
    },
)

early_mtl = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=20, restore_best_weights=True
)

history_mtl = multitask_model.fit(
    train_ds_mtl,
    epochs=200,
    validation_data=test_ds_mtl,
    callbacks=[early_mtl],
    verbose=0,
)

results = multitask_model.evaluate(test_ds_mtl, verbose=0, return_dict=True)
print(f"Home MAE: {results['home_goals_mae']:.3f}")
print(f"Away MAE: {results['away_goals_mae']:.3f}")


In [ ]:
sample_predictions = multitask_model.predict(X_test_mtl[:5], verbose=0)
for idx, (home_pred, away_pred) in enumerate(
    zip(
        sample_predictions["home_goals"].squeeze(),
        sample_predictions["away_goals"].squeeze(),
    ),
    start=1,
):
    print(
        f"Match {idx}: predicted {home_pred:.2f}-{away_pred:.2f}, "
        f"actual {int(y_home_test[idx-1])}-{int(y_away_test[idx-1])}"
    )


## Notes

This is the only Chapter 7 exception to the one-notebook-per-chapter rule.
The main notebook holds the PyTorch-centered walkthrough, while this
companion notebook keeps the TensorFlow implementation fully runnable and
separate.
